# License Plate Detection — Model 3: RetinaNet
**CMPS 261 — Machine Learning Project**

RetinaNet is a single-stage detector built on a Feature Pyramid Network (FPN) backbone:
- **FPN** — extracts multi-scale features so the model handles plates at different distances
- **Focal loss** — down-weights easy background examples so the model focuses on hard ones
- **Two-phase training** — freeze backbone → train head, then unfreeze all → fine-tune end-to-end

Training uses early stopping on **validation F1** rather than validation loss, which correlates better with detection accuracy on small datasets.

## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/archive'):
        if not os.path.exists(zip_path):
            raise RuntimeError('license_plate_data.zip not found in Google Drive root.')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset archive.')
    BASE_DIR = '/content'
else:
    BASE_DIR = '..'

def prepare_purified_dataset(base_dir, seed=42):
    """Create a deduplicated YOLO split from data/archive."""
    import hashlib
    import random
    import shutil
    import xml.etree.ElementTree as ET
    from pathlib import Path

    base_dir = Path(base_dir)
    img_dir = base_dir / 'data' / 'archive' / 'images'
    ann_dir = base_dir / 'data' / 'archive' / 'annotations'
    yolo_dir = base_dir / 'data' / 'yolo'

    if not img_dir.exists() or not ann_dir.exists():
        raise RuntimeError(f'Raw dataset not found under {base_dir / "data" / "archive"}')

    def file_md5(path):
        h = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    def parse_xml(xml_path):
        root = ET.parse(xml_path).getroot()
        filename = root.find('filename').text
        img_w = int(root.find('size/width').text)
        img_h = int(root.find('size/height').text)
        boxes = []
        for obj in root.findall('object'):
            boxes.append((
                int(obj.find('bndbox/xmin').text),
                int(obj.find('bndbox/ymin').text),
                int(obj.find('bndbox/xmax').text),
                int(obj.find('bndbox/ymax').text),
            ))
        return filename, img_w, img_h, boxes

    def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        return cx, cy, w, h

    xml_files = sorted(ann_dir.glob('*.xml'))
    hash_to_xmls = {}
    for xml_path in xml_files:
        filename, *_ = parse_xml(xml_path)
        img_path = img_dir / filename
        if img_path.exists():
            hash_to_xmls.setdefault(file_md5(img_path), []).append(xml_path)

    unique_xmls = sorted(min(group) for group in hash_to_xmls.values())
    print(f'Dedup: {len(xml_files)} XMLs -> {len(unique_xmls)} unique images ({len(xml_files) - len(unique_xmls)} duplicate copies removed)')

    random.seed(seed)
    random.shuffle(unique_xmls)
    n = len(unique_xmls)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    splits = {
        'train': unique_xmls[:n_train],
        'val': unique_xmls[n_train:n_train + n_val],
        'test': unique_xmls[n_train + n_val:],
    }

    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)
    for split in splits:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for xml_path in files:
            filename, img_w, img_h, boxes = parse_xml(xml_path)
            shutil.copy2(img_dir / filename, yolo_dir / 'images' / split / filename)
            label_path = yolo_dir / 'labels' / split / f'{Path(filename).stem}.txt'
            with open(label_path, 'w') as f:
                for xmin, ymin, xmax, ymax in boxes:
                    cx, cy, w, h = voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
                    f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

    yaml_path = yolo_dir / 'dataset.yaml'
    yaml_root = str(yolo_dir.resolve()) if str(base_dir) != '/content' else '/content/data/yolo'
    with open(yaml_path, 'w') as f:
        f.write(f'path: {yaml_root}\n')
        f.write('train: images/train\n')
        f.write('val:   images/val\n')
        f.write('test:  images/test\n\n')
        f.write('nc: 1\n')
        f.write("names: ['licence']\n")

    seen = {}
    for split in splits:
        for img in (yolo_dir / 'images' / split).iterdir():
            h = file_md5(img)
            if h in seen and seen[h] != split:
                raise RuntimeError(f'Cross-split duplicate after dedup: {img.name} in {split} matches {seen[h]}')
            seen[h] = split

    print('Data prepared:')
    for split, files in splits.items():
        print(f'  {split:<5}: {len(files)} images')
    print(f'  YAML  : {yaml_path}')
    print('  Cross-split duplicates: 0 (verified)')
    return str(yaml_path)

YAML_PATH = prepare_purified_dataset(BASE_DIR)

import json, random, time

if IN_COLAB:
    from google.colab import files
    TRAIN_IMG = '/content/data/yolo/images/train'
    VAL_IMG   = '/content/data/yolo/images/val'
    TEST_IMG  = '/content/data/yolo/images/test'
    TRAIN_LBL = '/content/data/yolo/labels/train'
    VAL_LBL   = '/content/data/yolo/labels/val'
    TEST_LBL  = '/content/data/yolo/labels/test'
    MODEL_SAVE_PATH = '/content/retinanet_best.pth'
    RESULTS_DIR = '/content/results'
else:
    TRAIN_IMG = '../data/yolo/images/train'
    VAL_IMG   = '../data/yolo/images/val'
    TEST_IMG  = '../data/yolo/images/test'
    TRAIN_LBL = '../data/yolo/labels/train'
    VAL_LBL   = '../data/yolo/labels/val'
    TEST_LBL  = '../data/yolo/labels/test'
    MODEL_SAVE_PATH = '../models/retinanet_best.pth'
    RESULTS_DIR = '../results'

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

import torch
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())

DEVICE = (torch.device('cuda') if torch.cuda.is_available() else
          torch.device('mps') if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f'Device: {DEVICE}' + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))


## 2. Dataset & DataLoaders

Reads the pre-split YOLO `.txt` labels and converts normalised `cxcywh` to absolute `xyxy`. Training split uses colour jitter, horizontal flip, random scale and random crop augmentation.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import torch
import os

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.augment = augment
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'): continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx - w/2) * W);  ymin = max(0.0, (cy - h/2) * H)
                xmax = min(float(W), (cx + w/2) * W); ymax = min(float(H), (cy + h/2) * H)
                if xmax > xmin and ymax > ymin: boxes.append([xmin, ymin, xmax, ymax])
        if not boxes: boxes = [[0.0, 0.0, 1.0, 1.0]]

        if self.augment:
            if random.random() > 0.5:
                img = TF.hflip(img)
                W2  = img.size[0]
                boxes = [[W2-b[2], b[1], W2-b[0], b[3]] for b in boxes]
            img = TF.adjust_brightness(img, 1 + random.uniform(-0.4, 0.4))
            img = TF.adjust_contrast(img,   1 + random.uniform(-0.4, 0.4))
            img = TF.adjust_saturation(img, 1 + random.uniform(-0.3, 0.3))
            img = TF.adjust_hue(img,        random.uniform(-0.08, 0.08))
            if random.random() > 0.4:
                scale = random.uniform(0.75, 1.0)
                new_W, new_H = int(W*scale), int(H*scale)
                img = TF.resize(img, (new_H, new_W))
                pad_x = random.randint(0, W-new_W); pad_y = random.randint(0, H-new_H)
                img = TF.pad(img, (pad_x, pad_y, W-new_W-pad_x, H-new_H-pad_y))
                boxes = [[b[0]*scale+pad_x, b[1]*scale+pad_y,
                          b[2]*scale+pad_x, b[3]*scale+pad_y] for b in boxes]
            if random.random() > 0.5:
                crop_scale = random.uniform(0.85, 1.0)
                cW, cH = int(W*crop_scale), int(H*crop_scale)
                x0 = random.randint(0, W-cW); y0 = random.randint(0, H-cH)
                img = TF.resize(TF.crop(img, y0, x0, cH, cW), (H, W))
                nb = [[max(0.0,(b[0]-x0)/crop_scale), max(0.0,(b[1]-y0)/crop_scale),
                       min(float(W),(b[2]-x0)/crop_scale), min(float(H),(b[3]-y0)/crop_scale)]
                      for b in boxes]
                boxes = [b for b in nb if b[2]>b[0] and b[3]>b[1]] or boxes

        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return TF.to_tensor(img), {'boxes': boxes, 'labels': labels}

def collate_fn(batch): return tuple(zip(*batch))

train_ds = LicensePlateDataset(TRAIN_IMG, TRAIN_LBL, augment=True)
val_ds   = LicensePlateDataset(VAL_IMG,   VAL_LBL,   augment=False)
test_ds  = LicensePlateDataset(TEST_IMG,  TEST_LBL,  augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

## 3. Build the Model

Pretrained RetinaNet ResNet50-FPN v2. We replace only the classification head to output 2 classes (background + licence plate).

In [ ]:
import torch
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
model   = retinanet_resnet50_fpn_v2(weights=weights)
num_anchors = model.head.classification_head.num_anchors
in_channels = model.head.classification_head.conv[0][0].in_channels
model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels, num_anchors=num_anchors, num_classes=2,
    norm_layer=torch.nn.BatchNorm2d,
)
model.to(DEVICE)
print('Model ready.')

## 4. Helper Functions

In [ ]:
import numpy as np
import torch

def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0, xB-xA) * max(0, yB-yA)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)

def eval_f1(loader, threshold=0.45):
    """Fast per-epoch F1 used for early stopping."""
    model.eval()
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(DEVICE) for img in images]
            preds  = model(images)
            for pred, target in zip(preds, targets):
                gt_boxes   = target['boxes'].numpy()
                pred_boxes = pred['boxes'][pred['scores'] >= threshold].cpu().numpy()
                matched = set()
                for pb in pred_boxes:
                    best_iou, best_j = 0, -1
                    for j, gb in enumerate(gt_boxes):
                        iou = compute_iou(pb, gb)
                        if iou > best_iou: best_iou, best_j = iou, j
                    if best_iou >= 0.5 and best_j not in matched:
                        tp += 1; matched.add(best_j)
                    else:
                        fp += 1
                fn += len(gt_boxes) - len(matched)
    p = tp / (tp + fp + 1e-6); r = tp / (tp + fn + 1e-6)
    return 2*p*r / (p + r + 1e-6)

def collect_predictions(loader):
    """Run best checkpoint once and cache predictions/targets for threshold search."""
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True))
    model.eval()
    cache = []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Collecting predictions', leave=False):
            images = [img.to(DEVICE) for img in images]
            preds = model(images)
            for pred, target in zip(preds, targets):
                scores = pred['scores'].detach().cpu().numpy()
                order = np.argsort(-scores)
                cache.append({
                    'boxes': pred['boxes'].detach().cpu().numpy()[order],
                    'scores': scores[order],
                    'gt_boxes': target['boxes'].numpy(),
                })
    return cache

def evaluate_cached(cache, threshold):
    tp, fp, fn = 0, 0, 0; iou_scores = []
    for item in cache:
        gt_boxes = item['gt_boxes']
        pred_boxes = item['boxes'][item['scores'] >= threshold]
        matched = set()
        for pb in pred_boxes:
            best_iou, best_j = 0, -1
            for j, gb in enumerate(gt_boxes):
                if j in matched:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= 0.5 and best_j != -1:
                tp += 1; matched.add(best_j); iou_scores.append(best_iou)
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched)
    p = tp/(tp+fp+1e-6); r = tp/(tp+fn+1e-6)
    f1 = 2*p*r/(p+r+1e-6)
    return p, r, f1, float(np.mean(iou_scores)) if iou_scores else 0.0

def find_best_threshold(cache, min_threshold=0.05, max_threshold=0.99):
    score_arrays = [item['scores'] for item in cache if len(item['scores'])]
    if not score_arrays:
        return min_threshold, evaluate_cached(cache, min_threshold)
    scores = np.concatenate(score_arrays)
    candidates = np.unique(scores[(scores >= min_threshold) & (scores <= max_threshold)])
    candidates = np.unique(np.concatenate(([min_threshold, max_threshold], candidates)))
    best_thresh, best_metrics = min_threshold, evaluate_cached(cache, min_threshold)
    for thresh in candidates:
        metrics = evaluate_cached(cache, float(thresh))
        if (metrics[2], metrics[0], float(thresh)) > (best_metrics[2], best_metrics[0], best_thresh):
            best_thresh, best_metrics = float(thresh), metrics
    return best_thresh, best_metrics

def evaluate(loader, threshold):
    """Full evaluation: loads best checkpoint, returns P/R/F1/mean-IoU."""
    return evaluate_cached(collect_predictions(loader), threshold)

## 5. Training Loop

Defines `run_phase` — reusable for both phases. Tracks and returns per-epoch train loss and val F1 for plotting.

In [ ]:
import time
import torch
from tqdm import tqdm

def run_phase(model, train_loader, val_loader, lr, max_epochs, patience,
              phase_name, use_cosine=False):
    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0005)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=lr/20)
                 if use_cosine else
                 torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3))

    best_f1 = 0.0; no_improve = 0
    train_losses, val_f1s = [], []

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        model.train()
        total = 0
        for images, targets in tqdm(train_loader, desc=f'[{phase_name}] E{epoch}', leave=False):
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss    = sum(model(images, targets).values())
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step(); total += loss.item()
        train_loss = total / len(train_loader)
        train_losses.append(train_loss)

        val_f1 = eval_f1(val_loader, threshold=0.45)
        val_f1s.append(val_f1)

        if use_cosine: scheduler.step()
        else:          scheduler.step(train_loss)

        flag = ''
        if val_f1 > best_f1:
            best_f1 = val_f1; no_improve = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH); flag = ' <- best'
        else:
            no_improve += 1

        print(f'[{phase_name}] Epoch {epoch:2d}/{max_epochs} | '
              f'Train: {train_loss:.4f} | Val F1: {val_f1:.4f} | '
              f'LR: {optimizer.param_groups[0]["lr"]:.2e} | '
              f'{time.time()-t0:.0f}s{flag}')

        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch}'); break

    return train_losses, val_f1s

### Phase 1 — Head Only

Backbone frozen. Only the classification and regression heads are trained.

In [ ]:
print('--- Phase 1: Head only ---')
for p in model.backbone.parameters(): p.requires_grad = False

p1_losses, p1_f1s = run_phase(
    model, train_loader, val_loader,
    lr=1e-3, max_epochs=40, patience=10,
    phase_name='Phase1', use_cosine=False
)

### Phase 2 — Full Fine-Tune

All weights unfrozen. Fine-tuned end-to-end starting from the best Phase 1 checkpoint.

In [ ]:
print('--- Phase 2: Full fine-tune ---')
for p in model.backbone.parameters(): p.requires_grad = True
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True))

p2_losses, p2_f1s = run_phase(
    model, train_loader, val_loader,
    lr=1e-4, max_epochs=80, patience=15,
    phase_name='Phase2', use_cosine=True
)

print('Training complete!')

## 6. Training Curves

In [ ]:
import os
import matplotlib.pyplot as plt

required = ['p1_losses', 'p2_losses', 'p1_f1s', 'p2_f1s']
missing = [name for name in required if name not in globals()]

if missing:
    print(f'Skipping training curves because training did not finish in this runtime. Missing: {missing}')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    all_losses = p1_losses + p2_losses
    all_f1s    = p1_f1s    + p2_f1s
    split      = len(p1_losses)

    ax1.plot(range(1, split+1), all_losses[:split], label='Phase 1')
    ax1.plot(range(split+1, len(all_losses)+1), all_losses[split:], label='Phase 2')
    ax1.axvline(split, color='gray', linestyle='--', alpha=0.5, label='Phase boundary')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss')
    ax1.set_title('RetinaNet - Training Loss'); ax1.legend()

    ax2.plot(range(1, split+1), all_f1s[:split], label='Phase 1')
    ax2.plot(range(split+1, len(all_f1s)+1), all_f1s[split:], label='Phase 2')
    ax2.axvline(split, color='gray', linestyle='--', alpha=0.5, label='Phase boundary')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val F1')
    ax2.set_title('RetinaNet - Validation F1'); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'retinanet_loss_curve.png'), dpi=150)
    plt.show()
    print('Saved: results/retinanet_loss_curve.png')


## 7. Find Best Threshold & Evaluate on Test Set

Search the actual validation prediction scores for the threshold that maximizes F1, then run final evaluation once on the held-out **test** set.

In [ ]:
import numpy as np
import torch

print('Finding exact best RetinaNet threshold on validation scores...')
val_cache = collect_predictions(val_loader)
best_thresh, val_metrics = find_best_threshold(val_cache)
print(f'Best validation F1 threshold: {best_thresh:.4f}')
print(f'Val P={val_metrics[0]:.4f} R={val_metrics[1]:.4f} F1={val_metrics[2]:.4f}')

test_cache = collect_predictions(test_loader)
precision, recall, f1, mean_iou = evaluate_cached(test_cache, best_thresh)

print(f'\n=== Test Results (validation-tuned threshold={best_thresh:.4f}) ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

## 8. Visualise Predictions

In [ ]:
import os
import random
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if 'best_thresh' not in globals():
    best_thresh = 0.45
    print(f'best_thresh not found; using fallback threshold {best_thresh}')

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True))
model.eval()
sample_indices = random.sample(range(len(test_ds)), min(8, len(test_ds)))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, sample_indices):
        img_tensor, target = test_ds[idx]
        pred = model([img_tensor.to(DEVICE)])[0]

        img_np = img_tensor.permute(1, 2, 0).numpy()
        ax.imshow(img_np)

        for box in target['boxes']:
            x1,y1,x2,y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='red', facecolor='none'))

        for box, score in zip(pred['boxes'], pred['scores']):
            if score < best_thresh: continue
            x1,y1,x2,y2 = box.cpu().tolist()
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='lime', facecolor='none'))
            ax.text(x1, y1-4, f'{score:.2f}', color='lime', fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))
        ax.axis('off')

for ax in axes[len(sample_indices):]:
    ax.axis('off')

plt.suptitle('RetinaNet - Test Predictions (red=GT, lime=pred)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'retinanet_predictions.png'), dpi=150)
plt.show()
print('Saved: results/retinanet_predictions.png')


## 9. Save Metrics for Comparison

In [ ]:
import os
import json

required = ['best_thresh', 'precision', 'recall', 'f1', 'mean_iou']
missing = [name for name in required if name not in globals()]

if missing:
    print(f'Skipping metric JSON save because evaluation did not finish. Missing: {missing}')
else:
    retinanet_metrics = {
        'model'    : 'RetinaNet (ResNet50-FPN v2)',
        'threshold': round(float(best_thresh), 4),
        'precision': round(precision, 4),
        'recall'   : round(recall,    4),
        'f1'       : round(f1,        4),
        'mean_iou' : round(mean_iou,  4),
    }

    with open(os.path.join(RESULTS_DIR, 'retinanet_metrics.json'), 'w') as f:
        json.dump(retinanet_metrics, f, indent=2)

    print('Metrics saved to results/retinanet_metrics.json')
    print(json.dumps(retinanet_metrics, indent=2))


## 10. Download Weights (Colab only)

In [ ]:
import os

if IN_COLAB:
    from google.colab import files
    if os.path.exists(MODEL_SAVE_PATH):
        files.download(MODEL_SAVE_PATH)
    else:
        print(f'Weight file not found yet: {MODEL_SAVE_PATH}')

    metrics_path = os.path.join(RESULTS_DIR, 'retinanet_metrics.json')
    if os.path.exists(metrics_path):
        files.download(metrics_path)
    else:
        print(f'Metrics file not found yet: {metrics_path}')
    print('Download step finished.')
else:
    print(f'Weights saved at: {MODEL_SAVE_PATH}')
